# Interactive Topology Optimization

Pick a problem type and its size/volume-fraction, pick a model and its
hyperparameters, then run the cell below the controls. A progress line is
printed every 25 steps while the optimization runs, and the final run
produces:

- **Compliance** (final/best objective value)
- **Gray fraction** (how far the design is from a crisp 0/1 material layout)
- **Topology PNG** (saved to `interactive_outputs/` and shown inline)
- **Elapsed time** for the optimization run

**Instructions**
1. Run the *Setup* cell once per kernel session.
2. Run the *Controls* cell to display the widgets.
3. Adjust the problem/model controls, then click **Run Optimization**.
4. Re-click the button any time after changing a control to re-run.

## Problems

Problem choices are the exact preconfigured cases already defined in
`neural_structural_optimization.problems.PROBLEMS_BY_CATEGORY` (the same set
listed in `problems.txt`) — pick a **Category** (e.g. `mbb_beam`, `l_shape_0.4`,
`thin_support_bridge`) and then a specific **Config** (its fixed
width x height and target density). There are no free-form size/density
sliders: each config's dimensions are already known to work with every model
below.

## Models

- **Pixel - LBFGS / MMA / OC**: direct per-pixel density, optimized with
  L-BFGS, the Method of Moving Asymptotes (`nlopt`), or Optimality Criteria.
  No extra hyperparameters.
- **CNN - LBFGS**: a convolutional generator (`CNNModel`) reparameterizes the
  density field from a latent vector. Configure `latent_size` and
  `dense_channels`.
- **Hybrid KAN - LBFGS**: `HybridKANModel` — same CNN decoder as above, but a
  KAN learns per-channel gating from the latent vector. Configure
  `latent_size`, `hidden_size`, `num_kan_layers`, `grid`, `spline order (k)`.
- **Base KAN - LBFGS**: `BaseKANModel` — a coordinate KAN mapping `(x, y)`
  directly to density. Configure hidden layer widths, `grid`, `spline order (k)`.

Note: CNN and Hybrid KAN use a fixed 4x spatial upsampling internally, so
`width` and `height` must both be divisible by 4 — true for every config in
`problems.py`, so this is handled automatically.

In [3]:
import sys

try:
    import ipywidgets  # noqa: F401
except ImportError:
    get_ipython().system(f"{sys.executable} -m pip install -q ipywidgets")

# Force the inline backend so figures always render as output in this
# notebook instead of popping up in a separate GUI window.
get_ipython().run_line_magic("matplotlib", "inline")

# Auto-reload local modules (models.py, problems.py, ...) on every cell run so
# edits to those files take effect without restarting the kernel.
get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")

import os
import threading
import time
import traceback

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.ioff()  # never auto-open a GUI window; figures are shown via display()

from neural_structural_optimization import problems, topo_api
from models import (
    PixelModel,
    CNNModel,
    HybridKANModel,
    BaseKANModel,
    train_lbfgs,
    method_of_moving_asymptotes,
    optimality_criteria,
)

OUTPUT_DIR = "interactive_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROGRESS_EVERY = 25  # print a loss update every this many optimization steps

# Fixed CNN / Hybrid-KAN spatial decoder shape: 4x total upsampling so that
# any width/height divisible by 4 works out of the box.
DECODER_RESIZES = (1, 2, 2, 1)
DECODER_CONV_FILTERS = (64, 32, 16, 1)

CATEGORIES = sorted(problems.PROBLEMS_BY_CATEGORY)

MODEL_OPTIONS = [
    "Pixel - LBFGS",
    "Pixel - MMA",
    "Pixel - OC",
    "CNN - LBFGS",
    "Hybrid KAN - LBFGS",
    "Base KAN - LBFGS",
]

print("Setup complete: {} problem categories ({} configs), {} models available.".format(
    len(CATEGORIES), len(problems.PROBLEMS_BY_NAME), len(MODEL_OPTIONS)))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Setup complete: 28 problem categories (116 configs), 6 models available.


In [4]:
# --- Interactive Controls ---
style = {"description_width": "initial"}

def _config_options(category):
    """(label, name) pairs for every preset config in this category,
    e.g. label='96 x 32, density=0.5' -> name='mbb_beam_96x32_0.5'."""
    return [
        ("{} x {}, density={}".format(p.width, p.height, p.density), p.name)
        for p in problems.PROBLEMS_BY_CATEGORY[category]
    ]


problem_category = widgets.Dropdown(
    options=CATEGORIES,
    value=CATEGORIES[0],
    description="Category:",
    style=style,
)
problem_config = widgets.Dropdown(
    options=_config_options(problem_category.value),
    description="Config:",
    style=style,
)


def on_category_change(change):
    problem_config.options = _config_options(change["new"])


problem_category.observe(on_category_change, names="value")

model_name = widgets.Dropdown(options=MODEL_OPTIONS, value="Pixel - LBFGS", description="Model:", style=style)

# Max step count: a slider for quick adjustment, linked to a type-in box so
# an exact value can be entered directly.
max_iterations = widgets.IntSlider(value=100, min=10, max=2000, step=10, description="Max steps:", style=style)
max_iterations_text = widgets.BoundedIntText(value=100, min=1, max=100000, description="(exact):", style=style, layout=widgets.Layout(width="160px"))
widgets.jslink((max_iterations, "value"), (max_iterations_text, "value"))
max_steps_box = widgets.HBox([max_iterations, max_iterations_text])

# CNN params
cnn_latent_size = widgets.IntSlider(value=128, min=16, max=256, step=16, description="Latent size:", style=style)
cnn_dense_channels = widgets.IntSlider(value=32, min=8, max=64, step=8, description="Dense channels:", style=style)
cnn_box = widgets.VBox([widgets.HTML("<b>CNN parameters</b>"), cnn_latent_size, cnn_dense_channels])

# Hybrid KAN params
hy_latent_size = widgets.IntSlider(value=128, min=16, max=256, step=16, description="Latent size:", style=style)
hy_hidden_size = widgets.IntSlider(value=64, min=8, max=128, step=8, description="Hidden size:", style=style)
hy_num_kan_layers = widgets.IntSlider(value=1, min=1, max=3, step=1, description="KAN layers:", style=style)
hy_grid = widgets.IntSlider(value=5, min=2, max=20, step=1, description="Grid size:", style=style)
hy_k = widgets.IntSlider(value=3, min=1, max=5, step=1, description="Spline order (k):", style=style)
hybrid_box = widgets.VBox([
    widgets.HTML("<b>Hybrid KAN parameters</b>"),
    hy_latent_size, hy_hidden_size, hy_num_kan_layers, hy_grid, hy_k,
])

# Base KAN params
kan_layers_text = widgets.Text(value="16, 16", description="Hidden layers:", style=style)
kan_grid = widgets.IntSlider(value=8, min=2, max=100, step=1, description="Grid size:", style=style)
kan_k = widgets.IntSlider(value=3, min=1, max=5, step=1, description="Spline order (k):", style=style)
basekan_box = widgets.VBox([
    widgets.HTML("<b>Base KAN parameters</b>"),
    kan_layers_text, kan_grid, kan_k,
])

param_boxes = {
    "CNN - LBFGS": cnn_box,
    "Hybrid KAN - LBFGS": hybrid_box,
    "Base KAN - LBFGS": basekan_box,
}


def on_model_change(change):
    selected = change["new"]
    for name, box in param_boxes.items():
        box.layout.display = "flex" if name == selected else "none"


model_name.observe(on_model_change, names="value")

run_button = widgets.Button(description="Run Optimization", button_style="success", icon="play")
output = widgets.Output()

# Some notebook front-ends (observed in VS Code's Jupyter extension) can
# dispatch a single button click as two overlapping callback invocations,
# which interleave their prints. Guard with a non-blocking lock so any
# duplicate/overlapping invocation is dropped immediately instead of running
# concurrently with the first.
_run_lock = threading.Lock()


def run_optimization(_button):
    if not _run_lock.acquire(blocking=False):
        return  # a run is already in progress (likely a duplicate click event)
    run_button.disabled = True
    try:
      with output:
        clear_output(wait=True)
        try:
            p_name = problem_config.value
            problem = problems.PROBLEMS_BY_NAME[p_name]
            args = topo_api.specified_task(problem)

            m_name = model_name.value
            iters = max_iterations.value

            print("Problem: {} ({})".format(p_name, problem_category.value))
            print("Domain: {} x {}, target density: {}".format(problem.width, problem.height, problem.density))
            print("Model: {}, Max steps: {}".format(m_name, iters))

            needs_decoder = m_name in ("CNN - LBFGS", "Hybrid KAN - LBFGS")
            total_resize = int(np.prod(DECODER_RESIZES))
            if needs_decoder and (problem.width % total_resize or problem.height % total_resize):
                raise ValueError(
                    "{} requires width and height to both be divisible by {} "
                    "(got {}x{}). Pick a different config.".format(
                        m_name, total_resize, problem.width, problem.height)
                )

            print("Optimizing...")
            t0 = time.time()
            if m_name == "Pixel - LBFGS":
                model = PixelModel(seed=0, args=args)
                ds = train_lbfgs(model, iters, progress_every=PROGRESS_EVERY)
            elif m_name == "Pixel - MMA":
                model = PixelModel(seed=0, args=args)
                ds = method_of_moving_asymptotes(model, iters, progress_every=PROGRESS_EVERY)
            elif m_name == "Pixel - OC":
                model = PixelModel(seed=0, args=args)
                ds = optimality_criteria(model, iters, progress_every=PROGRESS_EVERY)
            elif m_name == "CNN - LBFGS":
                model = CNNModel(
                    seed=0, args=args,
                    latent_size=cnn_latent_size.value,
                    dense_channels=cnn_dense_channels.value,
                    resizes=DECODER_RESIZES,
                    conv_filters=DECODER_CONV_FILTERS,
                )
                ds = train_lbfgs(model, iters, progress_every=PROGRESS_EVERY)
            elif m_name == "Hybrid KAN - LBFGS":
                model = HybridKANModel(
                    seed=0, args=args,
                    latent_size=hy_latent_size.value,
                    hidden_size=hy_hidden_size.value,
                    num_kan_layers=hy_num_kan_layers.value,
                    grid=hy_grid.value,
                    k=hy_k.value,
                    resizes=DECODER_RESIZES,
                    conv_filters=DECODER_CONV_FILTERS,
                )
                ds = train_lbfgs(model, iters, progress_every=PROGRESS_EVERY)
            elif m_name == "Base KAN - LBFGS":
                layers = tuple(int(x.strip()) for x in kan_layers_text.value.split(",") if x.strip())
                model = BaseKANModel(
                    seed=0, args=args,
                    kan_layers=layers,
                    grid=kan_grid.value,
                    k=kan_k.value,
                )
                ds = train_lbfgs(model, iters, progress_every=PROGRESS_EVERY)
            else:
                raise ValueError("Unknown model option: {}".format(m_name))
            elapsed = time.time() - t0

            losses = ds.loss.values
            valid_mask = ~np.isnan(losses)
            best_step = int(np.nanargmin(losses))
            compliance = float(losses[valid_mask].min())

            final_design = np.clip(ds.design.isel(step=best_step).values, 0.0, 1.0)
            gray_fraction = float(np.mean(4.0 * final_design * (1.0 - final_design)))

            safe_model = m_name.replace(" ", "_").replace("/", "_")
            png_path = os.path.join(OUTPUT_DIR, "{}_{}.png".format(p_name, safe_model))
            plt.imsave(png_path, 1.0 - final_design, cmap="gray", vmin=0.0, vmax=1.0)

            print()
            print("=" * 55)
            print("Compliance (best loss):  {:.4f}  (step {})".format(compliance, best_step))
            print("Gray fraction:           {:.4f}".format(gray_fraction))
            print("Elapsed time:            {:.2f} s".format(elapsed))
            print("Topology PNG saved to:   {}".format(png_path))
            print("=" * 55)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

            ax1.imshow(1.0 - final_design, cmap="gray", vmin=0.0, vmax=1.0)
            ax1.set_title("Final design: {} / {}".format(p_name, m_name))
            ax1.axis("off")

            ax2.plot(np.arange(len(losses)), losses)
            ax2.axvline(best_step, color="red", linestyle="--", linewidth=1, label="best step")
            ax2.set_title("Compliance vs. iteration")
            ax2.set_xlabel("Iteration")
            ax2.set_ylabel("Compliance")
            ax2.legend()
            ax2.grid(True)

            plt.tight_layout()
            display(fig)   # render inline in the notebook output, never a popup window
            plt.close(fig)
        except Exception:
            traceback.print_exc()
    finally:
        run_button.disabled = False
        _run_lock.release()


run_button.on_click(run_optimization)

# Initial layout
on_model_change({"new": model_name.value})

display(widgets.VBox([
    widgets.HTML("<h3>Problem parameters</h3>"),
    problem_category, problem_config,
    widgets.HTML("<hr><h3>Model parameters</h3>"),
    model_name, max_steps_box,
    cnn_box, hybrid_box, basekan_box,
    run_button,
]))
display(output)


Output()